# Load data for data folder

In [10]:
import pandas as pd
import numpy as np


df = pd.read_csv("../data/RAA04_Estimates_of_Household_Income.csv")


# Show unique Statistic Label values

In [4]:

df["Statistic Label"].unique()

array(["Compensation of Employees (i.e. Wages and Salaries, Benefits in kind, Employers' social insurance contribution)",
       'Rent of dwellings (including imputed rent of owner-occupied dwellings)',
       'Total Household Income', 'Total Income per Person',
       'Index of Total Income per Person', 'Current Taxes on Income',
       'Disposable Household Income', 'Disposable Income per Person',
       'Index of Disposable Income per Person',
       'Disposable Income per Person (excluding Rent)',
       'Index of Disposable Income per Person (excluding Rent)'],
      dtype=object)

# Statistic Label to horizontal attributes

In [ ]:

df = pd.read_csv("../data/RAA04_Estimates_of_Household_Income.csv")


# Pivot: Statistic Label to horizontal attributes
wide_df = df.pivot_table(
    index=["Year", "NUTS 2 Region", "UNIT"],   # keep these vertical
    columns="Statistic Label",                 # turn into horizontal columns
    values="VALUE",                            # fill with numeric values
    aggfunc="first"
)

# Reset index to make it a flat table
wide_df = wide_df.reset_index()

# Save
wide_df.to_csv("../data/RAA04_wide_format.csv", index=False)

# Smaller data size ： Merge Rows: One Row per Year + Region

In [13]:
# Load the existing wide-format dataset
df = pd.read_csv("../data/RAA04_wide_format.csv")

# Metadata columns (these should NOT be merged across rows)
meta = ["Year", "NUTS 2 Region", "UNIT"]

# Value columns (all columns that are actual data and need merging)
value_cols = [c for c in df.columns if c not in meta]

print("Value columns detected:")
value_cols

Value columns detected:


["Compensation of Employees (i.e. Wages and Salaries, Benefits in kind, Employers' social insurance contribution)",
 'Current Taxes on Income',
 'Disposable Household Income',
 'Disposable Income per Person',
 'Disposable Income per Person (excluding Rent)',
 'Index of Disposable Income per Person',
 'Index of Disposable Income per Person (excluding Rent)',
 'Index of Total Income per Person',
 'Rent of dwellings (including imputed rent of owner-occupied dwellings)',
 'Total Household Income',
 'Total Income per Person']

In [14]:
# Mapping for UNIT suffix
def unit_suffix(u):
    if "Million" in u:
        return "(Million)"
    elif "State" in u or "100" in u:
        return "(State100)"
    else:
        return "(Euro)"   # Default unit suffix

In [15]:
# Group data by Year + Region (each group contains 3 UNIT rows)
grouped = df.groupby(["Year", "NUTS 2 Region"])

final_rows = []

for (year, region), group in grouped:
    # Start building the merged single row
    row = {"Year": year, "NUTS 2 Region": region}
    
    # For each economic indicator column
    for col in value_cols:
        
        # Find the row where this column has a non-null value
        non_null_rows = group[group[col].notna()]
        
        # If no data at all for this column, skip it
        if len(non_null_rows) == 0:
            continue
        
        # Extract the actual value
        val = non_null_rows.iloc[0][col]
        
        # Identify which UNIT this value came from
        unit = non_null_rows.iloc[0]["UNIT"]
        
        # Rename column with unit suffix for clarity
        new_col = col + " " + unit_suffix(unit)
        
        # Store into row dictionary
        row[new_col] = val
    
    final_rows.append(row)


In [16]:
# Convert merged rows into DataFrame
final = pd.DataFrame(final_rows)

# Save final result
final.to_csv("../data/RAA04_final_one_row_with_rename.csv", index=False)

final.head()

,Year,NUTS 2 Region,"Compensation of Employees (i.e. Wages and Salaries, Benefits in kind, Employers' social insurance contribution) (Million)",Current Taxes on Income (Million),Disposable Household Income (Million),Disposable Income per Person (Euro),Disposable Income per Person (excluding Rent) (Euro),Index of Disposable Income per Person (State100),Index of Disposable Income per Person (excluding Rent) (State100),Index of Total Income per Person (State100),Rent of dwellings (including imputed rent of owner-occupied dwellings) (Million),Total Household Income (Million),Total Income per Person (Euro)
0,2000,Eastern & Midland,23144.07,11861.73,23233.55,12873.23,11863.41,102.58,48.05,110.73,1822.52,35095.28,19445.58
1,2000,Ireland,42023.36,19066.96,47739.92,12549.09,11713.17,100.00,100.00,100.00,3180.03,66806.88,17561.10
2,2000,Northern & Western,5945.04,2240.21,7965.59,11533.87,10874.85,91.91,16.85,84.15,455.13,10205.80,14777.61
3,2000,Southern,12934.25,4965.02,16540.78,12637.81,11948.36,100.71,35.10,93.57,902.38,21505.80,16431.28
4,2001,Eastern & Midland,25819.84,12795.37,26760.07,14592.59,13382.46,102.73,48.09,110.69,2219.14,39555.44,21570.05


# Filier for target year range

In [20]:
df = pd.read_csv("../data/RAA04_final_one_row_with_rename.csv")
df = df[(df["Year"] >= 2014) & (df["Year"] <= 2023)]
df.to_csv("../data/RAA04_final_2014_2023.csv", index=False)
df.head(40)


,Year,NUTS 2 Region,"Compensation of Employees (i.e. Wages and Salaries, Benefits in kind, Employers' social insurance contribution) (Million)",Current Taxes on Income (Million),Disposable Household Income (Million),Disposable Income per Person (Euro),Disposable Income per Person (excluding Rent) (Euro),Index of Disposable Income per Person (State100),Index of Disposable Income per Person (excluding Rent) (State100),Index of Total Income per Person (State100),Rent of dwellings (including imputed rent of owner-occupied dwellings) (Million),Total Household Income (Million),Total Income per Person (Euro)
56,2014,Eastern & Midland,41611.18,23636.15,40361.61,18015.50,16413.23,98.72,46.91,106.99,3589.69,63997.76,28565.55
57,2014,Ireland,76019.28,39320.23,84899.19,18248.23,16848.99,100.00,100.00,100.00,6509.95,124219.42,26699.73
58,2014,Northern & Western,11090.48,4886.56,14850.63,17490.51,16345.64,95.85,17.70,87.06,972.07,19737.19,23245.72
59,2014,Southern,23317.62,10797.52,29686.95,18993.45,17747.02,104.08,35.39,97.01,1948.18,40484.47,25901.60
60,2015,Eastern & Midland,44490.29,25006.63,43140.00,19078.03,17298.34,99.58,47.32,107.70,4024.31,68146.63,30136.85
61,2015,Ireland,80874.94,41439.60,89960.96,19157.86,17603.23,100.00,100.00,100.00,7300.20,131400.56,27982.73
62,2015,Northern & Western,11689.94,5135.14,15545.97,18140.57,16867.53,94.69,17.49,86.24,1090.96,20681.11,24132.76
63,2015,Southern,24694.72,11297.83,31275.00,19824.90,18439.90,103.48,35.19,96.44,2184.93,42572.83,26986.48
64,2016,Eastern & Midland,47593.13,25523.50,46127.43,20155.84,18219.24,101.31,47.92,108.68,4431.98,71650.93,31308.59
65,2016,Ireland,86257.55,42360.93,94553.66,19895.71,18309.99,100.00,100.00,100.00,7536.10,136914.59,28809.18
